# 48-hour observation sensitivity, calibration, gate interventions, ON/OFF statistics

This notebook contains code selected from the authors’ research notebooks because its analysis is described in the accepted COST manuscript. Outputs and execution counts have been removed. It may require authorized datasets, saved model artifacts, and variables established by preceding pipeline notebooks. See `docs/EXECUTION_ORDER.md`.


# 48-hour observation sensitivity, calibration, gate interventions, ON/OFF statistics

This notebook documents one experiment reported in the COST manuscript.


In [ ]:
# CELL 1 — SETUP, PATHS, FLAGS
import os, sys, json, math, time, random, warnings, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running in Colab (or Drive already mounted):", e)

BASE       = Path(os.environ.get("COST_BASE", "/content/drive/MyDrive/COST_PROJECT"))
MIMIC_BASE = BASE / "mimic-iv-2.1"
HOSP       = MIMIC_BASE / "hosp"
ICU        = MIMIC_BASE / "icu"
CACHE_FILE = MIMIC_BASE / "mimic_cache" / "hf_cohort_processed.npz"
PRED_DIR   = MIMIC_BASE / "saved_predictions"
SAVE_DIR   = MIMIC_BASE / "saved_models"
REV_DIR    = MIMIC_BASE / "revision2_outputs"
CKPT_MAIN  = SAVE_DIR / "dsrnn_v2_best.pt"
ID_FILE    = REV_DIR / "REV6_FINAL_exact_identifiers.csv"
EICU_PRED  = REV_DIR / "external_eicu" / "eicu_external_predictions.npz"
OUT        = REV_DIR / "final_round_reviewer"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- FLAGS ----------------
RUN_COHORT_AUDIT         = True   # A1/A2: reads admissions.csv + icustays.csv (small, read-only)
RUN_CHECKPOINT_INFERENCE = True   # A4-A6: rebuild preprocessing + inference with frozen checkpoints
RUN_GATING_SEED_RECAL    = True   # A5: also recalibrate gating ON/OFF seed checkpoints (if *_best.pt exist)
RUN_EICU                 = True   # A8: only runs if EICU_PRED exists
RUN_OPTIONAL_TRAINING    = False  # A9: NEW TRAINING (≈1–2 h GPU). Keep False unless you decide to add it.
N_BOOT   = 2000
SEED     = 42
TRAIN_YEARS, VAL_YEARS, TEST_YEARS = range(2008, 2016), range(2016, 2018), range(2018, 2020)

VITAL_SIGNS = ['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'spo2', 'temperature']
LAB_VALUES  = ['creatinine', 'bnp', 'lactate', 'troponin', 'sodium', 'potassium', 'chloride',
               'bicarbonate', 'bun', 'glucose', 'hemoglobin', 'wbc', 'platelets', 'inr', 'pt',
               'ptt', 'ph', 'pao2', 'paco2', 'fio2', 'albumin', 'bilirubin_total', 'alt', 'ast']
STATIC_FEATURES = ['age', 'gender_male', 'admission_emergency', 'admission_urgent', 'elixhauser_score',
                   'hypertension', 'diabetes', 'ckd', 'copd', 'af', 'prior_hf']
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES, N_STATIC = len(ALL_TEMPORAL_FEATURES), len(STATIC_FEATURES)
N_INPUT = N_FEATURES * 3

REPORT = {}      # key numbers for the response letter
FLAGS  = []      # audit notes to verify against the final pipeline

# Saved test predictions to evaluate: display name -> (file in PRED_DIR, key). Edit if your final files differ.
PRED_FILES = {
    "Logistic Regression": ("baseline_lr_predictions.npz", "probs"),
    "Random Forest":       ("baseline_rf_predictions.npz", "probs"),
    "Standard LSTM":       ("baseline_lstm_predictions.npz", "probs"),
    "Attention LSTM":      ("baseline_attn_lstm_predictions.npz", "probs"),
    "TCN":                 ("baseline_tcn_predictions.npz", "probs"),
    "COST (Initial)":      ("cost_before_predictions.npz", "probs"),
    "COST (Improved) single":   ("cost_improved_predictions.npz", "probs_single"),
    "COST (Improved) ensemble": ("cost_improved_predictions.npz", "probs_ensemble"),
}

GATING_SEEDS = [42, 43, 44]   # Revision II gating ON/OFF artefacts in REV_DIR (skipped if absent)

random.seed(SEED); np.random.seed(SEED)
for p in [CACHE_FILE, PRED_DIR, ID_FILE, CKPT_MAIN]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")
print("Output folder:", OUT)

In [ ]:
# CELL 2 — UTILITIES: metrics, DeLong, (cluster) bootstrap, calibration, recalibration, plots
def show(df, n=60):
    try:
        display(df.head(n))          # noqa: F821  (Colab/Jupyter)
    except NameError:
        print(df.head(n).to_string())

def save_table(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False)
    TABLES[name] = df
    return df
TABLES = {}

def sens_at_spec(y, p, target_spec=0.90):
    # identical definition to the manuscript code
    fpr, tpr, _ = roc_curve(y, p)
    idx = np.argmin(np.abs((1 - fpr) - target_spec))
    return float(tpr[idx])

def _clip(p): return np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
def logit(p): p = _clip(p); return np.log(p / (1 - p))
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def fit_logistic(X, y, offset=None, iters=100):
    # Newton-Raphson logistic regression; X must include an intercept column if wanted
    X = np.asarray(X, float); y = np.asarray(y, float)
    off = np.zeros(len(y)) if offset is None else np.asarray(offset, float)
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        mu = sigmoid(X @ b + off)
        W = mu * (1 - mu)
        H = X.T @ (X * W[:, None]) + 1e-9 * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - mu))
        b += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return b

def ece_score(y, p, n_bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def calib_stats(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); z = logit(p)
    prev = y.mean()
    slope = fit_logistic(np.c_[np.ones_like(z), z], y)[1]
    citl  = fit_logistic(np.ones((len(y), 1)), y, offset=z)[0]   # calibration-in-the-large
    brier = brier_score_loss(y, p)
    return {"mean_pred": p.mean(), "observed": prev, "citl_intercept": citl, "calib_slope": slope,
            "ECE": ece_score(y, p), "Brier": brier, "Brier_ref": prev * (1 - prev),
            "scaled_Brier": 1 - brier / (prev * (1 - prev))}

def full_metrics(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    d = {"N": len(y), "deaths": int(y.sum()),
         "AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p),
         "Sens@90Spec": sens_at_spec(y, p)}
    d.update(calib_stats(y, p))
    return d

# ---- DeLong (fast, Sun & Xu 2014) ----
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    out = np.empty(N); out[J] = T + 1
    return out

def delong_paired(y, p1, p2):
    y = np.asarray(y).astype(int)
    order = np.argsort(-y, kind="mergesort")      # positives first
    m = int(y.sum()); n = len(y) - m
    P = np.vstack([np.asarray(p1, float), np.asarray(p2, float)])[:, order]
    tx = np.vstack([_midrank(r[:m]) for r in P]); ty = np.vstack([_midrank(r[m:]) for r in P])
    tz = np.vstack([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    l = np.array([1, -1]); var = float(l @ cov @ l)
    zstat = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else np.nan
    pval = 2 * stats.norm.sf(abs(zstat)) if var > 0 else np.nan
    return float(aucs[0]), float(aucs[1]), float(zstat), float(pval)

# ---- Bootstrap (optionally clustered by subject_id) ----
def _boot_indices(n, clusters, rng):
    if clusters is None:
        return rng.integers(0, n, n)
    uniq = np.unique(clusters)
    pick = rng.choice(uniq, size=len(uniq), replace=True)
    pos = {c: np.flatnonzero(clusters == c) for c in uniq}
    return np.concatenate([pos[c] for c in pick])

def boot_ci(y, p, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); vals = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        vals.append(fn(y[idx], p[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, p1, p2, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); d = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        d.append(fn(y[idx], p1[idx]) - fn(y[idx], p2[idx]))
    d = np.asarray(d)
    p_two = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float(min(p_two, 1.0))

# ---- Recalibration ----
def prior_shift(p, target_prev, train_prev=0.5):
    # analytic correction: model trained with a class-balanced sampler (effective prior 0.5)
    return sigmoid(logit(p) + np.log(target_prev / (1 - target_prev)) - np.log(train_prev / (1 - train_prev)))

class Platt:
    def fit(self, p, y):
        z = logit(p); self.b = fit_logistic(np.c_[np.ones_like(z), z], y); return self
    def predict(self, p):
        z = logit(p); return sigmoid(self.b[0] + self.b[1] * z)

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(np.asarray(p), np.asarray(y)); return self
    def predict(self, p):
        return np.clip(self.m.predict(np.asarray(p)), 1e-6, 1 - 1e-6)

class InterceptOnly:
    def fit(self, p, y):
        self.a = fit_logistic(np.ones((len(y), 1)), y, offset=logit(p))[0]; return self
    def predict(self, p):
        return sigmoid(logit(p) + self.a)

def crossfit_recal(y, p, maker, n_splits=5, n_repeats=20, seed=SEED):
    # out-of-fold recalibration on the evaluation set itself (supplementary; no information leak per fold)
    y = np.asarray(y).astype(int); p = np.asarray(p, float); rows = []; oof0 = None
    for r in range(n_repeats):
        oof = np.zeros_like(p)
        for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed + r).split(p, y):
            oof[te] = maker().fit(p[tr], y[tr]).predict(p[te])
        rows.append(calib_stats(y, oof))
        if oof0 is None:
            oof0 = oof
    return pd.DataFrame(rows).mean().to_dict(), oof0

def reliability_plot(curves, y, title, fname, n_bins=10):
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    for name, p in curves.items():
        p = np.asarray(p); bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        xs, ys = [], []
        for b in range(n_bins):
            m = idx == b
            if m.sum() >= 5:
                xs.append(p[m].mean()); ys.append(np.asarray(y)[m].mean())
        ax.plot(xs, ys, "o-", ms=4, label=f"{name} (Brier={brier_score_loss(y, p):.3f})")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed mortality")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(title, fontsize=10); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / fname, dpi=300); plt.show(); plt.close(fig)

print("Utilities ready")

## A0 — Load artefacts & cohort audit

Memuat cache MIMIC (urutan pasien yang dipakai semua model), file identitas `REV6_FINAL_exact_identifiers.csv`
(hasil rekonstruksi REV-6J yang sudah terbukti identik dengan cache: N, urutan label, dan urutan tahun),
serta semua prediksi test yang tersimpan. Bagian ini menghitung jumlah subject unik dan overlap antar split **pada artefak yang sedang dipakai**,
untuk dibandingkan dengan Table VI/XIV. Jika hasil naskah berasal dari file cohort lain, ganti path di CELL 1.

In [ ]:
# CELL 3 — LOAD CACHE, IDENTIFIERS, SAVED PREDICTIONS + ALIGNMENT CHECKS
cache = np.load(CACHE_FILE, allow_pickle=True)
X_temporal_raw = cache["X_temporal"].astype(np.float32)
X_static_raw   = cache["X_static"].astype(np.float32)
labels_all     = cache["labels"].astype(int)
years_all      = cache["years"].astype(int)
N_ALL = len(labels_all)
print("Cache:", X_temporal_raw.shape, X_static_raw.shape, "deaths", labels_all.sum())

HAS_IDS = ID_FILE.exists()
if HAS_IDS:
    ids = pd.read_csv(ID_FILE)
    HAS_IDS = (len(ids) == N_ALL and np.array_equal(ids["label"].to_numpy().astype(int), labels_all)
               and np.array_equal(ids["real_admit_year"].to_numpy().astype(int), years_all))
    print("Identifier file aligned with cache order:", "OK" if HAS_IDS else "NO — subject-level analyses will be skipped")
if not HAS_IDS:
    print("No aligned identifier file: building placeholder IDs (A0/A1 subject & 48h analyses skipped).")
    ids = pd.DataFrame({"subject_id": np.arange(N_ALL), "hadm_id": np.arange(N_ALL), "stay_id": np.arange(N_ALL),
                        "real_admit_year": years_all, "label": labels_all})
    RUN_COHORT_AUDIT = False

split_of = np.where(np.isin(years_all, list(TRAIN_YEARS)), "train",
           np.where(np.isin(years_all, list(VAL_YEARS)), "val",
           np.where(np.isin(years_all, list(TEST_YEARS)), "test", "none")))
ids["split"] = split_of
test_mask, val_mask, train_mask = split_of == "test", split_of == "val", split_of == "train"
ids_test = ids[test_mask].reset_index(drop=True)
y_test = ids_test["label"].to_numpy().astype(int)
print({s: int((split_of == s).sum()) for s in ["train", "val", "test"]})

PREDS = {}
for name, (fn, key) in PRED_FILES.items():
    f = PRED_DIR / fn
    if not f.exists():
        print("missing:", fn); continue
    d = np.load(f, allow_pickle=True)
    assert np.array_equal(d["test_labels"].astype(int), y_test), f"{fn}: test label/order mismatch"
    PREDS[name] = d[key].astype(np.float64)

GATING = {}   # (arm, seed) -> dict(single, ensemble)
for arm in ["ON", "OFF"]:
    for s in GATING_SEEDS:
        f = REV_DIR / f"gating_{arm}_seed{s}_predictions.npz"
        if f.exists():
            d = np.load(f, allow_pickle=True)
            assert np.array_equal(d["test_labels"].astype(int), y_test)
            GATING[(arm, s)] = {"single": d["probs_single"].astype(float), "ensemble": d["probs_ensemble"].astype(float)}
print("Loaded predictions:", list(PREDS), "| gating artefacts:", sorted(GATING))
show(pd.DataFrame([{"Model": k, "AUROC": roc_auc_score(y_test, v), "AUPRC": average_precision_score(y_test, v),
                    "Brier": brier_score_loss(y_test, v), "min p": v.min(), "max p": v.max(), "mean p": v.mean()}
                   for k, v in PREDS.items()]).round(4))

In [ ]:
# CELL 4 — A0: COHORT / SPLIT AUDIT (what the artefacts actually contain)
subj = {s: set(ids.loc[split_of == s, "subject_id"]) for s in ["train", "val", "test"]}
rows = []
for s in ["train", "val", "test"]:
    m = split_of == s
    rows.append({"Split": s, "Records (ICU stays)": int(m.sum()), "Unique subjects": len(subj[s]),
                 "Repeated-subject records": int(m.sum() - len(subj[s])),
                 "Deaths": int(labels_all[m].sum()), "Mortality %": 100 * labels_all[m].mean()})
cohort_tab = save_table(pd.DataFrame(rows).round(2), "A0_split_unique_subjects")
show(cohort_tab)

ov = pd.DataFrame([
    {"Pair": "train–val",  "Shared subjects": len(subj["train"] & subj["val"])},
    {"Pair": "train–test", "Shared subjects": len(subj["train"] & subj["test"])},
    {"Pair": "val–test",   "Shared subjects": len(subj["val"] & subj["test"])},
])
save_table(ov, "A0_subject_overlap"); show(ov)

n_unique_all = ids["subject_id"].nunique()
test_seen = ids_test["subject_id"].isin(subj["train"] | subj["val"]).to_numpy()
REPORT.update({"unique_subjects_total": int(n_unique_all), "records_total": int(N_ALL),
               "overlap_train_val": int(ov.iloc[0, 1]), "overlap_train_test": int(ov.iloc[1, 1]),
               "overlap_val_test": int(ov.iloc[2, 1]),
               "test_records_with_subject_seen_in_train_or_val": int(test_seen.sum())})
print(f"Unique subjects in final cohort: {n_unique_all:,} of {N_ALL:,} records")
if HAS_IDS and (n_unique_all != N_ALL or ov["Shared subjects"].sum() > 0):
    FLAGS.append(
        f"[CHECK] The identifier file aligned with this cache contains {n_unique_all:,} unique subjects for {N_ALL:,} records, "
        f"with shared subjects train–val={ov.iloc[0,1]}, train–test={ov.iloc[1,1]}, val–test={ov.iloc[2,1]}. "
        f"Compare with Table VI/XIV and Sec. III-B (one stay per subject_id, 0 overlap). If the reported results came from a "
        f"different (subject-level) cohort file, point CACHE_FILE/ID_FILE to it and re-run; otherwise use the A2 "
        f"subject-disjoint results.")
for f in FLAGS: print(f)

## A1 — Jendela observasi 48 jam vs outcome ()

Kohort memakai ICU LOS > 24 jam, tetapi input mencakup 48 jam pertama sejak ICU `intime`.
Pasien yang meninggal sebelum jam ke-48 punya sekuens yang terpotong; pola "data berhenti" dapat
membocorkan label. Bagian ini menghitung:
- kematian < 24 jam, 24–48 jam, dan ≥ 48 jam sejak ICU intime;
- ICU discharge dan hospital discharge sebelum jam ke-48;
- seberapa prediktif "jam observasi terakhir" saja terhadap mortalitas (indikator kebocoran).

In [ ]:
# CELL 5 — A1: 48-HOUR WINDOW AUDIT (reads admissions + icustays only)
if RUN_COHORT_AUDIT:
    adm = pd.read_csv(HOSP / "admissions.csv", usecols=["hadm_id", "dischtime", "deathtime", "hospital_expire_flag"],
                      parse_dates=["dischtime", "deathtime"])
    icu = pd.read_csv(ICU / "icustays.csv", usecols=["stay_id", "intime", "outtime"], parse_dates=["intime", "outtime"])
    tl = ids.merge(icu, on="stay_id", how="left").merge(adm, on="hadm_id", how="left")
    assert len(tl) == N_ALL and tl["intime"].notna().all()
    death_time = tl["deathtime"].where(tl["deathtime"].notna(), tl["dischtime"])   # fallback for expired w/o deathtime
    tl["h_death"]     = np.where(tl["label"] == 1, (death_time - tl["intime"]).dt.total_seconds() / 3600, np.nan)
    tl["h_icu_out"]   = (tl["outtime"] - tl["intime"]).dt.total_seconds() / 3600
    tl["h_hosp_out"]  = (tl["dischtime"] - tl["intime"]).dt.total_seconds() / 3600
    tl["death_lt24"]  = tl["h_death"] < 24
    tl["death_24_48"] = (tl["h_death"] >= 24) & (tl["h_death"] < 48)
    tl["death_lt48"]  = tl["h_death"] < 48
    tl["icu_out_lt48"]  = tl["h_icu_out"] < 48
    tl["hosp_out_lt48"] = tl["h_hosp_out"] < 48

    # observed-data truncation from the raw (pre-imputation) cache
    obs_v = ~np.isnan(X_temporal_raw[:, :, :len(VITAL_SIGNS)])
    any_obs = obs_v.any(axis=2)
    tl["last_vital_hour"] = np.where(any_obs.any(1), 47 - np.argmax(any_obs[:, ::-1], axis=1), -1)
    tl["n_vital_hours"]   = any_obs.sum(1)

    rows = []
    for s in ["train", "val", "test", "all"]:
        d = tl if s == "all" else tl[tl["split"] == s]
        dd = d[d["label"] == 1]
        rows.append({"Split": s, "N": len(d), "Deaths": int(d["label"].sum()),
                     "Death <24h": int(d["death_lt24"].sum()), "Death 24–48h": int(d["death_24_48"].sum()),
                     "Death <48h (% of deaths)": f"{int(d['death_lt48'].sum())} ({100*d['death_lt48'].sum()/max(len(dd),1):.1f}%)",
                     "ICU discharge <48h": int(d["icu_out_lt48"].sum()),
                     "Hospital discharge alive <48h": int((d["hosp_out_lt48"] & (d["label"] == 0)).sum()),
                     "Median h to death": float(np.nanmedian(dd["h_death"])) if len(dd) else np.nan})
    win_tab = save_table(pd.DataFrame(rows), "A1_48h_window_counts")
    show(win_tab)

    tt = tl[tl["split"] == "test"].reset_index(drop=True)
    leak = pd.DataFrame([
        {"Proxy feature (test set)": "last observed vital hour (lower = earlier stop)",
         "AUROC for mortality": roc_auc_score(tt["label"], -tt["last_vital_hour"])},
        {"Proxy feature (test set)": "number of hours with any vital",
         "AUROC for mortality": roc_auc_score(tt["label"], -tt["n_vital_hours"])},
    ])
    save_table(leak, "A1_truncation_leakage_proxy"); show(leak.round(4))
    t = win_tab.set_index("Split")
    REPORT.update({"test_deaths_lt48": int(tl.loc[tl.split == "test", "death_lt48"].sum()),
                   "all_deaths_lt48": int(tl["death_lt48"].sum()),
                   "test_icu_out_lt48": int(tl.loc[tl.split == "test", "icu_out_lt48"].sum()),
                   "truncation_proxy_auroc": float(leak.iloc[0, 1])})
else:
    tl = None
    print("A1 skipped")

## A2 — Sensitivity analysis performa test (tanpa retraining)

Prediksi tersimpan dievaluasi ulang pada subset test:
1. **Full test** (seperti di naskah);
2. **Landmark 48 jam** — buang pasien yang meninggal < 48 jam;
3. **Masih di ICU pada jam 48** — buang juga ICU discharge < 48 jam (sekuens lengkap);
4. **Subject-disjoint** — buang pasien test yang subject_id-nya muncul di train/val;
5. Kombinasi (3) + (4).

CI 95% memakai bootstrap yang di-*cluster* per `subject_id`; perbandingan COST vs LSTM memakai DeLong berpasangan.

In [ ]:
# CELL 6 — A2: SUBSET PERFORMANCE + PAIRED COMPARISONS
subsets = {"Full test": np.ones(len(y_test), bool), "Subject-disjoint test": ~test_seen}
if tl is not None:
    tt = tl[tl["split"] == "test"].reset_index(drop=True)
    assert np.array_equal(tt["stay_id"].to_numpy(), ids_test["stay_id"].to_numpy())
    subsets["Alive at 48h (landmark)"] = ~tt["death_lt48"].to_numpy()
    subsets["Alive & in ICU at 48h"]   = (~tt["death_lt48"] & ~tt["icu_out_lt48"]).to_numpy()
    subsets["Alive & in ICU at 48h, subject-disjoint"] = subsets["Alive & in ICU at 48h"] & ~test_seen
clusters_test = ids_test["subject_id"].to_numpy()

rows = []
for sname, m in subsets.items():
    yy = y_test[m]
    for model, p in PREDS.items():
        r = {"Subset": sname, "Model": model, "N": int(m.sum()), "Deaths": int(yy.sum()),
             "AUROC": roc_auc_score(yy, p[m]), "AUPRC": average_precision_score(yy, p[m]),
             "Brier": brier_score_loss(yy, p[m]), "Sens@90Spec": sens_at_spec(yy, p[m])}
        lo, hi = boot_ci(yy, p[m], clusters=clusters_test[m], n_boot=N_BOOT // 2)
        r["AUROC 95% CI"] = f"{lo:.3f}–{hi:.3f}"
        rows.append(r)
sub_tab = save_table(pd.DataFrame(rows), "A2_subset_performance")
show(sub_tab.round(4))

comp = []
pairs = [("COST (Improved) single", "Standard LSTM"), ("COST (Improved) ensemble", "Standard LSTM"),
         ("COST (Improved) ensemble", "Attention LSTM"), ("COST (Improved) ensemble", "COST (Initial)")]
for sname, m in subsets.items():
    for a, b in pairs:
        if a in PREDS and b in PREDS:
            auc_a, auc_b, z, p = delong_paired(y_test[m], PREDS[a][m], PREDS[b][m])
            md_, lo, hi, pb = boot_diff(y_test[m], PREDS[a][m], PREDS[b][m], clusters=clusters_test[m], n_boot=N_BOOT // 2)
            comp.append({"Subset": sname, "A": a, "B": b, "AUROC A": auc_a, "AUROC B": auc_b, "Δ": auc_a - auc_b,
                         "DeLong p": p, "Cluster-bootstrap 95% CI Δ": f"{lo:.4f} to {hi:.4f}", "Bootstrap p": pb})
comp_tab = save_table(pd.DataFrame(comp), "A2_paired_comparisons")
show(comp_tab.round(4))

r = sub_tab.set_index(["Subset", "Model"])
for key in subsets:
    if ("COST (Improved) ensemble" in PREDS):
        REPORT[f"AUROC_ens__{key}"] = float(r.loc[(key, "COST (Improved) ensemble"), "AUROC"])

## A3 — Kalibrasi semua model (prediksi test tersimpan) — Brier score tinggi

Prevalensi test = 15.5%, jadi model "tanpa informasi" yang selalu memprediksi prevalensi memberi
Brier ≈ 0.131 (`Brier_ref`). Model yang dilatih dengan `WeightedRandomSampler` (batch 50:50),
focal loss α = 0.75, dan label smoothing akan menggeser probabilitas ke atas (*prior shift*),
sehingga Brier tinggi walaupun AUROC baik. Diagnosis yang dihitung:
mean predicted vs observed, calibration-in-the-large (intercept), calibration slope, ECE, scaled Brier.

Perbaikan yang dilaporkan:
- **Prior-shift correction (analitik):** hanya memakai prevalensi training (13.0%), tanpa fitting pada outcome test.
- **Platt / isotonic cross-fitted pada test (5-fold × 20):** pelengkap, untuk model tanpa prediksi validation.
- Rekalibrasi utama berbasis **validation set** ada di A5.

In [ ]:
# CELL 7 — A3: CALIBRATION OF SAVED TEST PREDICTIONS
train_prev = float(labels_all[train_mask].mean())
REPORT["train_prevalence"] = train_prev; REPORT["test_prevalence"] = float(y_test.mean())
rows = []
for model, p in PREDS.items():
    r = {"Model": model, "Recalibration": "None (as reported)"}; r.update(calib_stats(y_test, p)); rows.append(r)
    if not model.startswith(("Logistic", "Random")):   # sklearn baselines were not trained with a balanced sampler
        r = {"Model": model, "Recalibration": f"Prior-shift (train prevalence {train_prev:.3f})"}
        r.update(calib_stats(y_test, prior_shift(p, train_prev))); rows.append(r)
    for nm, mk in [("Platt, cross-fitted on test", Platt), ("Isotonic, cross-fitted on test", Iso)]:
        st, _ = crossfit_recal(y_test, p, mk)
        r = {"Model": model, "Recalibration": nm}; r.update(st); rows.append(r)
cal_tab = save_table(pd.DataFrame(rows), "A3_calibration_saved_predictions")
show(cal_tab.round(4))

if "COST (Improved) ensemble" in PREDS:
    p = PREDS["COST (Improved) ensemble"]
    _, oof_platt = crossfit_recal(y_test, p, Platt, n_repeats=1)
    reliability_plot({"As reported": p, "Prior-shift": prior_shift(p, train_prev), "Platt (cross-fitted)": oof_platt},
                     y_test, "COST (Improved) ensemble — MIMIC-IV test", "A3_reliability_cost_improved.png")
    reliability_plot({k: PREDS[k] for k in ["Logistic Regression", "Standard LSTM", "Attention LSTM", "COST (Improved) ensemble"] if k in PREDS},
                     y_test, "As-reported probabilities, MIMIC-IV test", "A3_reliability_models.png")
    c = cal_tab[(cal_tab.Model == "COST (Improved) ensemble")].set_index("Recalibration")
    REPORT.update({"ens_brier_reported": float(c.iloc[0]["Brier"]), "ens_mean_pred": float(c.iloc[0]["mean_pred"]),
                   "ens_brier_prior_shift": float(c.iloc[1]["Brier"]), "brier_ref": float(c.iloc[0]["Brier_ref"])})

## A4 — Rekonstruksi preprocessing v2 (persis seperti training) + audit data

Kode `preprocess_v2` disalin dari notebook Master. Loop Python untuk mask/delta-t dan forward-fill diganti versi
vektorisasi yang **diverifikasi identik** dengan versi loop. Hasil test harus mereproduksi `probs_single`
yang tersimpan (AUROC 0.8002), sehingga inference validation set sah dipakai untuk rekalibrasi.

Audit yang ikut dicetak (perlu dicocokkan dengan teks naskah Section III-D/E dan Table XII):
- apakah lab masih punya NaN saat masuk `IterativeImputer` (di kode v2, `nan_to_num` dijalankan **sebelum** MICE);
- apakah missingness mask / delta-t untuk 24 variabel lab informatif atau konstan;
- distribusi suhu (itemid 223761 = °F dan 223762 = °C digabung dalam satu kanal).

In [ ]:
# CELL 8 — A4: EXACT PREPROCESSING V2 (vectorised, verified) + DATA AUDIT
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

def add_missingness_features_loop(X):          # original implementation (used only for verification)
    n, seq, feats = X.shape
    mask = (~np.isnan(X)).astype(np.float32)
    delta = np.zeros_like(X, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X[i, t, k]):
                    last_obs = t; delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)
    values = X.copy()
    for i in range(n):
        for k in range(len(VITAL_SIGNS)):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

def add_missingness_features(X):               # vectorised, identical output
    n, seq, feats = X.shape
    obs = ~np.isnan(X)
    mask = obs.astype(np.float32)
    t = np.arange(seq, dtype=np.int64)[None, :, None]
    last_obs = np.maximum.accumulate(np.where(obs, t, 0), axis=1)
    delta = np.where(obs, 0.0, (t - last_obs)).astype(np.float32) / np.float32(seq + 1e-8)
    values = X.copy()
    nv = len(VITAL_SIGNS)
    v = values[:, :, :nv]; ov = obs[:, :, :nv]
    idx = np.maximum.accumulate(np.where(ov, t[:, :, :1].repeat(nv, axis=2), -1), axis=1)
    filled = np.take_along_axis(np.nan_to_num(v, nan=0.0), np.clip(idx, 0, None), axis=1)
    values[:, :, :nv] = np.where(idx >= 0, filled, 0.0)
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

if RUN_CHECKPOINT_INFERENCE:
    # ---- verify vectorised == loop on a subset (after the same lab handling as training) ----
    sub = X_temporal_raw[:300].copy()
    sub[:, :, len(VITAL_SIGNS):] = np.nan_to_num(sub[:, :, len(VITAL_SIGNS):], nan=0.0, posinf=0.0, neginf=0.0)
    assert np.allclose(add_missingness_features(sub), add_missingness_features_loop(sub), atol=1e-7)
    print("Vectorised missingness features identical to original loop: OK")

    def preprocess_v2(X_temporal, X_static, labels, years):
        n_vital = len(VITAL_SIGNS); splits_raw = {}
        for name, yrs in [("train", TRAIN_YEARS), ("val", VAL_YEARS), ("test", TEST_YEARS)]:
            m = np.isin(years, list(yrs))
            splits_raw[name] = {"X_temp": X_temporal[m].copy(), "X_stat": X_static[m].copy(), "y": labels[m].copy()}
        audit = {"lab_nan_fraction_raw_train": float(np.isnan(splits_raw["train"]["X_temp"][:, :, n_vital:]).mean())}
        imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)
        tr = splits_raw["train"]["X_temp"]; n_tr, seq, n_feat = tr.shape
        lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
        lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)   # <- as in original code (before MICE)
        audit["lab_nan_fraction_entering_MICE"] = float(np.isnan(lab_train).mean())
        imp.fit(lab_train)
        for name in splits_raw:
            arr = splits_raw[name]["X_temp"]; n_sp = arr.shape[0]
            lab_d = np.nan_to_num(arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital), nan=0.0, posinf=0.0, neginf=0.0)
            out = imp.transform(lab_d)
            if name == "train":
                audit["MICE_max_abs_change_train"] = float(np.abs(out - lab_d).max())
            arr[:, :, n_vital:] = out.reshape(n_sp, seq, n_feat - n_vital)
        splits = {}
        for name in splits_raw:
            splits[name] = {"X_temp": add_missingness_features(splits_raw[name]["X_temp"]),
                            "X_stat": splits_raw[name]["X_stat"].copy(), "y": splits_raw[name]["y"].copy()}
        tr_flat = splits["train"]["X_temp"][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
        feat_mean = tr_flat.mean(axis=0); feat_std = tr_flat.std(axis=0) + 1e-8
        stat_mean = splits["train"]["X_stat"].mean(axis=0); stat_std = splits["train"]["X_stat"].std(axis=0) + 1e-8
        for name in splits:
            arr = splits[name]["X_temp"].copy()
            arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
            splits[name]["X_temp"] = arr
            splits[name]["X_stat"] = (splits[name]["X_stat"] - stat_mean) / stat_std
        norm = {"feat_mean": feat_mean, "feat_std": feat_std, "stat_mean": stat_mean, "stat_std": stat_std}
        return splits, norm, audit

    t0 = time.time()
    SPLITS, NORM, PREP_AUDIT = preprocess_v2(X_temporal_raw.copy(), X_static_raw.copy(), labels_all, years_all)
    print(f"preprocess_v2 rebuilt in {(time.time()-t0)/60:.1f} min")
    assert np.array_equal(SPLITS["test"]["y"].astype(int), y_test)

    tr = SPLITS["train"]["X_temp"]; nv = len(VITAL_SIGNS)
    lab_mask = tr[:, :, N_FEATURES + nv: 2 * N_FEATURES]; lab_delta = tr[:, :, 2 * N_FEATURES + nv:]
    vit_mask = tr[:, :, N_FEATURES: N_FEATURES + nv]
    temp = X_temporal_raw[:, :, VITAL_SIGNS.index("temperature")]; temp = temp[~np.isnan(temp)]
    PREP_AUDIT.update({
        "lab_mask_unique_values": np.unique(lab_mask).tolist(),
        "lab_delta_all_zero": bool(np.all(lab_delta == 0)),
        "vital_mask_observed_fraction": float(vit_mask.mean()),
        "temperature_values_below_45 (Celsius-like) %": float(100 * (temp < 45).mean()),
        "temperature_values_above_80 (Fahrenheit-like) %": float(100 * (temp > 80).mean()),
    })
    print(json.dumps(PREP_AUDIT, indent=2))
    pd.Series(PREP_AUDIT, dtype=object).to_csv(OUT / "A4_preprocessing_audit.csv")
    if PREP_AUDIT["lab_nan_fraction_entering_MICE"] == 0.0 and PREP_AUDIT["lab_delta_all_zero"]:
        FLAGS.append(
            "[CHECK] In the preprocess_v2 version reproduced here (validated against the checkpoint in A5), lab NaNs are zero-filled "
            f"before IterativeImputer (max MICE change {PREP_AUDIT['MICE_max_abs_change_train']:.2e}); lab masks are constant 1 and lab "
            "delta-t constant 0, so informative missingness/delta-t comes from the 7 vital signs. Verify against Sec. III-D/E, "
            "Table VIII and Table XII.")
    if PREP_AUDIT["temperature_values_below_45 (Celsius-like) %"] > 1 and PREP_AUDIT["temperature_values_above_80 (Fahrenheit-like) %"] > 1:
        FLAGS.append("[CHECK] Temperature channel mixes °F (itemid 223761) and °C (223762) values; mention as a limitation or harmonise in future work.")
    for f in FLAGS[-2:]: print(f)

## A5 — Rekalibrasi berbasis validation set (inference checkpoint beku)

Model utama di naskah = `dsrnn_v2_best.pt` (seed 44, snapshot terbaik). Kalibrator (Platt, isotonic) di-*fit*
pada **validation set (2016–2017)** lalu diterapkan ke **test (2018–2019)**; ini sesuai split kronologis naskah.
Jika checkpoint gating ON/OFF per seed (`gating_*_seed*_best.pt` dan `*_snapshots.pt`) tersedia, hal yang sama dilakukan
untuk single model dan snapshot ensemble.

In [ ]:
# CELL 9 — MODEL (state_dict-compatible with all COST Improved checkpoints) + INFERENCE HELPERS
if RUN_CHECKPOINT_INFERENCE:
    import torch, torch.nn as nn, torch.nn.functional as F
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)
    print("Device:", DEVICE)

    class SinusoidalPE(nn.Module):
        def __init__(self, d_model, max_len=256, dropout=0.1):
            super().__init__(); self.dropout = nn.Dropout(dropout)
            pe = torch.zeros(max_len, d_model); pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe.unsqueeze(0))
        def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])

    class StochasticDepth(nn.Module):
        def __init__(self, p=0.1): super().__init__(); self.p = p
        def forward(self, x, residual):
            if not self.training or self.p == 0: return x + residual
            keep = torch.bernoulli(torch.tensor(1 - self.p)).item(); return x + residual * keep

    class ClinicalSalienceV2(nn.Module):
        def __init__(self, input_dim, hidden_dim, n_features):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden_dim * 2); self.norm = nn.LayerNorm(hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, n_features); self.tau = nn.Parameter(torch.tensor(0.5))
        def forward(self, x_t, h_prev):
            h = self.fc1(torch.cat([x_t, h_prev], dim=-1)); h, gate = h.chunk(2, dim=-1)
            h = self.norm(h * torch.sigmoid(gate)); s_t = torch.sigmoid(self.fc2(h))
            m_hard = (s_t > self.tau).float(); return m_hard - s_t.detach() + s_t, s_t

    class TemporalMultiHeadAttn(nn.Module):
        def __init__(self, d_model, n_heads, dropout=0.1):
            super().__init__()
            self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model); self.sdrop = StochasticDepth(p=0.1)
        def forward(self, x):
            out, w = self.attn(x, x, x); return self.norm(self.sdrop(x, out)), w

    class StaticTemporalFusion(nn.Module):
        def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
            super().__init__()
            self.static_proj = nn.Linear(n_static, d_temporal)
            self.cross_attn = nn.MultiheadAttention(d_temporal, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_temporal)
            self.ffn = nn.Sequential(nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
                                     nn.Linear(d_temporal, d_out))
        def forward(self, s, h):
            q = self.static_proj(s).unsqueeze(1); ctx, w = self.cross_attn(q, h, h)
            ctx = self.norm(ctx.squeeze(1)); return self.ffn(torch.cat([ctx, h.mean(dim=1)], dim=-1)), w.squeeze(1)

    class COSTImproved(nn.Module):
        # gate_mode: 'learned' (paper, gating ON) | 'off' (bypass, gating OFF) |
        #            'all_zero' (m=0 → 0.1·z) | 'all_one' (m=1 → z) | 'random' (Bernoulli(rho)) | 'constant' ((0.1+0.9·rho)·z)
        def __init__(self, n_input=N_INPUT, n_static=N_STATIC, hidden_size=256, n_lstm_layers=2, n_heads=4,
                     attention_dim=128, salience_hidden=128, dropout=0.3, lambda_sparse=5e-4, gate_mode="learned", rho=0.0099):
            super().__init__()
            self.hidden_size, self.lambda_sparse, self.gate_mode, self.rho = hidden_size, lambda_sparse, gate_mode, rho
            self.input_proj = nn.Sequential(nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout))
            self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
            self.salience = ClinicalSalienceV2(hidden_size * 2, salience_hidden, hidden_size)
            self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers, batch_first=True, bidirectional=True,
                                dropout=dropout if n_lstm_layers > 1 else 0)
            self.bidir_proj = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.LayerNorm(hidden_size), nn.GELU())
            self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
            self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
            self.classifier = nn.Sequential(nn.LayerNorm(attention_dim), nn.Dropout(dropout),
                                            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
                                            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1))
            self._init_weights()
        def _init_weights(self):
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)
                elif isinstance(m, nn.LSTM):
                    for n_, p_ in m.named_parameters():
                        if "weight" in n_: nn.init.orthogonal_(p_)
                        elif "bias" in n_: nn.init.zeros_(p_)
        def forward(self, x_temporal, x_static, return_masks=False):
            b, seq, _ = x_temporal.shape
            z = self.pos_enc(self.input_proj(x_temporal))
            c = torch.zeros(b, self.hidden_size, device=x_temporal.device)
            gated, masks = [], []; sp = torch.tensor(0.0, device=x_temporal.device)
            for t in range(seq):
                z_t = z[:, t, :]
                if self.gate_mode == "off":
                    zg = z_t
                else:
                    if self.gate_mode == "learned":
                        m_t, _ = self.salience(z_t, c); sp = sp + m_t.norm(p=1)
                    elif self.gate_mode == "all_zero":
                        m_t = torch.zeros_like(z_t)
                    elif self.gate_mode == "all_one":
                        m_t = torch.ones_like(z_t)
                    elif self.gate_mode == "random":
                        m_t = (torch.rand_like(z_t) < self.rho).float()
                    elif self.gate_mode == "constant":
                        m_t = torch.full_like(z_t, self.rho)
                    else:
                        raise ValueError(self.gate_mode)
                    zg = m_t * z_t + (1 - m_t) * z_t * 0.1
                    if return_masks: masks.append(m_t.detach().unsqueeze(1))
                gated.append(zg.unsqueeze(1))
                c = c * 0.9 + zg.detach() * 0.1
            sp = self.lambda_sparse * sp / (seq * b) if self.gate_mode == "learned" else torch.tensor(0.0, device=x_temporal.device)
            lo, _ = self.lstm(torch.cat(gated, dim=1)); lo = self.bidir_proj(lo)
            H, _ = self.temporal_attn(lo); zz, _ = self.fusion(x_static, H)
            logits = self.classifier(zz).squeeze(-1)
            if return_masks:
                return logits, sp, (torch.cat(masks, dim=1) if masks else None)
            return logits, sp

    def load_state(path):
        obj = torch.load(path, map_location="cpu", weights_only=False)
        return obj["model_state_dict"] if isinstance(obj, dict) and "model_state_dict" in obj else obj

    def build_model(state, gate_mode="learned", rho=0.0099):
        m = COSTImproved(gate_mode=gate_mode, rho=rho).to(DEVICE)
        m.load_state_dict(state, strict=True); m.eval(); return m

    @torch.no_grad()
    def predict(model, split, batch=256, return_masks=False, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        X, S = SPLITS[split]["X_temp"], SPLITS[split]["X_stat"]; probs, masks = [], []
        for i in range(0, len(X), batch):
            xb = torch.from_numpy(X[i:i + batch]).float().to(DEVICE); sb = torch.from_numpy(S[i:i + batch]).float().to(DEVICE)
            if return_masks:
                lg, _, mk = model(xb, sb, return_masks=True); masks.append(mk.cpu().numpy().astype(np.uint8))
            else:
                lg, _ = model(xb, sb)
            probs.append(torch.sigmoid(lg).float().cpu().numpy())
        p = np.concatenate(probs).astype(np.float64)
        return (p, np.concatenate(masks)) if return_masks else p
    print("Model + inference helpers ready")

In [ ]:
# CELL 10 — A5: VALIDATION-BASED RECALIBRATION (main checkpoint + gating seeds)
if RUN_CHECKPOINT_INFERENCE:
    y_val = SPLITS["val"]["y"].astype(int)
    ckpt = torch.load(CKPT_MAIN, map_location="cpu", weights_only=False)
    for k in ["feat_mean", "feat_std", "stat_mean", "stat_std"]:
        assert np.allclose(np.asarray(ckpt["norm_stats"][k]), NORM[k], atol=1e-4), f"norm_stats mismatch: {k}"
    print("Rebuilt normalisation statistics match checkpoint: OK")
    MAIN = build_model(ckpt["model_state_dict"], "learned")
    print("Learned τ in checkpoint:", float(MAIN.salience.tau.detach().cpu()))
    REPORT["checkpoint_tau"] = float(MAIN.salience.tau.detach().cpu())

    p_test_main, M_TEST = predict(MAIN, "test", return_masks=True)
    p_val_main = predict(MAIN, "val")
    auc_rep = roc_auc_score(y_test, p_test_main)
    max_diff = float(np.abs(p_test_main - PREDS["COST (Improved) single"]).max()) if "COST (Improved) single" in PREDS else np.nan
    print(f"Reproduced test AUROC = {auc_rep:.4f} (saved 0.8002); max |Δp| vs saved probs_single = {max_diff:.2e}")
    REPORT.update({"reproduced_test_auroc": auc_rep, "reproduced_max_abs_diff": max_diff})
    REPRO_OK = bool(max_diff < 1e-3) or abs(auc_rep - roc_auc_score(y_test, PREDS.get("COST (Improved) single", p_test_main))) < 5e-4
    print("Checkpoint + rebuilt preprocessing reproduce the saved predictions:", "YES" if REPRO_OK else "NO")
    if not REPRO_OK:
        FLAGS.append("[CHECK] The checkpoint with the rebuilt preprocess_v2 does not reproduce the saved COST (Improved) "
                     "predictions; the checkpoint or preprocessing version differs. Treat A4–A6 results with caution.")

    def recal_rows(tag, p_val, p_tst):
        out = []
        for nm, fn in [("None (as reported)", lambda p: p),
                       (f"Prior-shift (train prevalence)", lambda p: prior_shift(p, train_prev)),
                       ("Platt fitted on validation", Platt().fit(p_val, y_val).predict),
                       ("Isotonic fitted on validation", Iso().fit(p_val, y_val).predict)]:
            pt = fn(p_tst); r = {"Model": tag, "Recalibration": nm, "AUROC": roc_auc_score(y_test, pt)}
            r.update(calib_stats(y_test, pt)); out.append(r)
        return out

    rows = recal_rows("COST (Improved) single — main checkpoint (seed 44)", p_val_main, p_test_main)
    VAL_PREDS = {"main_single": (p_val_main, p_test_main)}

    if RUN_GATING_SEED_RECAL:
        for arm in ["ON", "OFF"]:
            for s in [42, 43, 44]:
                fb, fs = REV_DIR / f"gating_{arm}_seed{s}_best.pt", REV_DIR / f"gating_{arm}_seed{s}_snapshots.pt"
                if not fb.exists():
                    continue
                mode = "learned" if arm == "ON" else "off"
                mdl = build_model(load_state(fb), mode)
                pv, pt = predict(mdl, "val"), predict(mdl, "test")
                if (arm, s) in GATING:
                    print(f"gating {arm} seed {s}: reproduced single AUROC {roc_auc_score(y_test, pt):.4f} "
                          f"(saved {roc_auc_score(y_test, GATING[(arm, s)]['single']):.4f})")
                rows += recal_rows(f"Gating {arm} seed {s} — single", pv, pt)
                VAL_PREDS[f"{arm}_{s}_single"] = (pv, pt)
                if fs.exists():
                    snaps = torch.load(fs, map_location="cpu", weights_only=False)
                    pvs, pts = [], []
                    for _, st in snaps:
                        m2 = build_model(st, mode); pvs.append(predict(m2, "val")); pts.append(predict(m2, "test"))
                    pv_e, pt_e = np.mean(pvs, 0), np.mean(pts, 0)
                    rows += recal_rows(f"Gating {arm} seed {s} — ensemble", pv_e, pt_e)
                    VAL_PREDS[f"{arm}_{s}_ensemble"] = (pv_e, pt_e)
                del mdl
                if DEVICE.type == "cuda": torch.cuda.empty_cache()

    recal_tab = save_table(pd.DataFrame(rows), "A5_validation_based_recalibration")
    show(recal_tab.round(4))
    platt_main = Platt().fit(p_val_main, y_val)
    reliability_plot({"As reported": p_test_main, "Prior-shift": prior_shift(p_test_main, train_prev),
                      "Platt (validation-fitted)": platt_main.predict(p_test_main),
                      "Isotonic (validation-fitted)": Iso().fit(p_val_main, y_val).predict(p_test_main)},
                     y_test, "COST (Improved) — recalibrated on validation, evaluated on test", "A5_reliability_validation_recal.png")
    r = recal_tab[recal_tab.Model.str.startswith("COST (Improved) single")].set_index("Recalibration")
    REPORT.update({"main_brier_reported": float(r.iloc[0]["Brier"]), "main_brier_prior_shift": float(r.iloc[1]["Brier"]),
                   "main_brier_platt_val": float(r.iloc[2]["Brier"]), "main_brier_iso_val": float(r.iloc[3]["Brier"]),
                   "main_slope_reported": float(r.iloc[0]["calib_slope"]), "main_slope_platt_val": float(r.iloc[2]["calib_slope"]),
                   "main_ECE_reported": float(r.iloc[0]["ECE"]), "main_ECE_platt_val": float(r.iloc[2]["ECE"]),
                   "platt_coefficients_main": platt_main.b.tolist()})

## A6 — Mekanisme controlled gating dan hubungannya dengan 99% latent sparsity ()

Formulasi yang **benar-benar diimplementasikan** di kode (untuk ditulis di Section III-H):

$$z_t = \mathrm{PE}\big(\mathrm{GELU}(\mathrm{LN}(W_{in}x_t + b_{in}))\big) \in \mathbb{R}^{256},\qquad x_t \in \mathbb{R}^{93}$$
$$u_t = W_a[z_t \,\|\, c_{t-1}] ,\quad v_t = W_b[z_t \,\|\, c_{t-1}],\quad s_t = \sigma\big(W_2\,\mathrm{LN}(u_t \odot \sigma(v_t)) + b_2\big)$$
$$m_t = \mathbb{1}[s_t > \tau]\ \ (\text{straight-through estimator}),\qquad \tau = 0.5$$
$$\tilde z_t = m_t \odot z_t + \gamma\,(1-m_t)\odot z_t = \big(\gamma + (1-\gamma)\,m_t\big)\odot z_t,\qquad \gamma = 0.1$$
$$c_t = 0.9\,c_{t-1} + 0.1\,\mathrm{sg}(\tilde z_t),\qquad c_0 = 0$$
$$\mathcal{L} = \mathcal{L}_{focal} + \alpha_{RD}\,\mathcal{L}_{R\text{-}Drop} + \lambda\,\frac{1}{TB}\sum_{t,b}\lVert m_t\rVert_1$$

Artinya, "binary latent sparsity 99%" = 99% dimensi laten **dilemahkan menjadi 10%**, bukan dibuang (*residual leak* γ = 0.1).
Karena itu komputasi tidak berkurang. Uji intervensi di bawah (inference saja, checkpoint beku) memeriksa apakah
1% dimensi yang aktif membawa informasi prediktif:
- `learned` (model seperti di naskah), `all_zero` (semua dilemahkan), `all_one` (tanpa pelemahan),
- `random` (mask acak dengan densitas sama, 10 ulangan), `constant` (skala seragam setara densitas).

In [ ]:
# CELL 11 — A6: MASK STATISTICS + GATE-INTERVENTION TEST (frozen main checkpoint)
if RUN_CHECKPOINT_INFERENCE:
    M = M_TEST.astype(np.float32)                         # (N_test, 48, 256)
    rho = float(M.mean())
    dim_freq = M.mean(axis=(0, 1)); t_freq = M.mean(axis=(0, 2)); pat_ret = M.mean(axis=(1, 2))
    gate_stats = {
        "retention_rate (fraction of m_t = 1)": rho, "binary latent sparsity %": 100 * (1 - rho),
        "latent dims never active on test": int((dim_freq == 0).sum()),
        "latent dims active >50% of time": int((dim_freq > 0.5).sum()),
        "dims accounting for 90% of activations": int(np.searchsorted(np.cumsum(np.sort(dim_freq)[::-1]) / max(dim_freq.sum(), 1e-12), 0.9) + 1),
        "retention at t=0 / t=47": f"{t_freq[0]:.4f} / {t_freq[-1]:.4f}",
        "AUROC of patient retention for mortality": roc_auc_score(y_test, pat_ret) if pat_ret.std() > 0 else np.nan,
        "Spearman(patient retention, predicted risk)": stats.spearmanr(pat_ret, p_test_main).correlation,
    }
    show(pd.DataFrame(gate_stats.items(), columns=["Statistic", "Value"]))
    pd.DataFrame(gate_stats.items(), columns=["Statistic", "Value"]).to_csv(OUT / "A6_gate_statistics.csv", index=False)
    REPORT.update({"gate_retention": rho, "gate_dims_never_active": gate_stats["latent dims never active on test"]})

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].bar(np.arange(256), np.sort(dim_freq)[::-1]); ax[0].set_title("Activation frequency per latent dimension (sorted)", fontsize=9)
    ax[0].set_xlabel("Latent dimension rank"); ax[0].set_ylabel("P(m=1)")
    ax[1].plot(t_freq); ax[1].set_title("Retention over the 48-h window", fontsize=9); ax[1].set_xlabel("Hour"); ax[1].set_ylabel("P(m=1)")
    fig.tight_layout(); fig.savefig(OUT / "A6_gate_activation.png", dpi=300); plt.show(); plt.close(fig)

    state = ckpt["model_state_dict"]; rows = []
    def add_row(name, p):
        a, b, z, pv = delong_paired(y_test, p_test_main, p)
        r = {"Gate intervention (inference only)": name, "AUROC": roc_auc_score(y_test, p),
             "AUPRC": average_precision_score(y_test, p), "Brier": brier_score_loss(y_test, p),
             "Sens@90Spec": sens_at_spec(y_test, p), "ΔAUROC vs learned": roc_auc_score(y_test, p) - a, "DeLong p vs learned": pv}
        rows.append(r)
    add_row("learned mask (as in paper)", p_test_main)
    add_row("all_zero: every dimension attenuated (0.1·z)", predict(build_model(state, "all_zero"), "test"))
    add_row("all_one: no attenuation (z)", predict(build_model(state, "all_one"), "test"))
    add_row(f"constant: uniform scale (0.1+0.9·ρ)·z, ρ={rho:.4f}", predict(build_model(state, "constant", rho), "test"))
    rnd = [predict(build_model(state, "random", rho), "test", seed=1000 + k) for k in range(10)]
    aucs = [roc_auc_score(y_test, p) for p in rnd]
    add_row("random mask, matched density (mean of 10 draws)", np.mean(rnd, 0))
    int_tab = save_table(pd.DataFrame(rows), "A6_gate_intervention_test")
    int_tab.loc[len(int_tab)] = {"Gate intervention (inference only)": "random mask: AUROC range over 10 draws",
                                 "AUROC": f"{min(aucs):.4f}–{max(aucs):.4f}"}
    show(int_tab.round(4))
    REPORT.update({"intervention_auroc_learned": float(rows[0]["AUROC"]), "intervention_auroc_all_zero": float(rows[1]["AUROC"]),
                   "intervention_auroc_all_one": float(rows[2]["AUROC"]), "intervention_auroc_constant": float(rows[3]["AUROC"]),
                   "intervention_auroc_random_range": [float(min(aucs)), float(max(aucs))]})

## A7 — Gating ON vs OFF (architecture-matched dense vs controlled gating) — statistik

Memakai prediksi tersimpan (seed 42/43/44, arsitektur dan training identik, hanya gating yang berbeda).
Dilaporkan per seed: ΔAUROC dengan DeLong berpasangan, ΔBrier, serta rata-rata Δ lintas seed dengan
bootstrap pasien berpasangan (pasien yang sama di-resample untuk ketiga seed).

In [ ]:
# CELL 12 — A7: GATING ON vs OFF
rows = []
for kind in ["single", "ensemble"]:
    for s in [42, 43, 44]:
        if ("ON", s) in GATING and ("OFF", s) in GATING:
            pon, poff = GATING[("ON", s)][kind], GATING[("OFF", s)][kind]
            a_on, a_off, z, p = delong_paired(y_test, pon, poff)
            rows.append({"Prediction": kind, "Seed": s, "AUROC ON": a_on, "AUROC OFF": a_off, "ΔAUROC": a_on - a_off,
                         "DeLong p": p, "AUPRC ON": average_precision_score(y_test, pon),
                         "AUPRC OFF": average_precision_score(y_test, poff),
                         "Brier ON": brier_score_loss(y_test, pon), "Brier OFF": brier_score_loss(y_test, poff),
                         "Sens@90 ON": sens_at_spec(y_test, pon), "Sens@90 OFF": sens_at_spec(y_test, poff),
                         "ECE ON": ece_score(y_test, pon), "ECE OFF": ece_score(y_test, poff)})
if rows:
    g_tab = save_table(pd.DataFrame(rows), "A7_gating_on_off_per_seed")
    show(g_tab.round(4))
    rng = np.random.default_rng(SEED); summ = []
    for kind in ["single", "ensemble"]:
        seeds = [s for s in [42, 43, 44] if ("ON", s) in GATING and ("OFF", s) in GATING]
        P_on = np.vstack([GATING[("ON", s)][kind] for s in seeds]); P_off = np.vstack([GATING[("OFF", s)][kind] for s in seeds])
        obs = np.mean([roc_auc_score(y_test, a) - roc_auc_score(y_test, b) for a, b in zip(P_on, P_off)])
        d = []
        for _ in range(N_BOOT):
            idx = _boot_indices(len(y_test), clusters_test, rng)
            if y_test[idx].min() == y_test[idx].max(): continue
            d.append(np.mean([roc_auc_score(y_test[idx], a[idx]) - roc_auc_score(y_test[idx], b[idx]) for a, b in zip(P_on, P_off)]))
        d = np.asarray(d)
        summ.append({"Prediction": kind, "Seeds": len(seeds), "Mean ΔAUROC (ON−OFF)": obs,
                     "95% CI (patient-cluster bootstrap)": f"{np.percentile(d, 2.5):.4f} to {np.percentile(d, 97.5):.4f}",
                     "Two-sided bootstrap p": float(min(1, 2 * min((d <= 0).mean(), (d >= 0).mean())))})
    gs = save_table(pd.DataFrame(summ), "A7_gating_on_off_summary")
    show(gs.round(4))
    REPORT["gating_on_off_summary"] = gs.to_dict(orient="records")
else:
    print("Gating ON/OFF prediction files not found in", REV_DIR)

## A8 — eICU: kalibrasi, rekalibrasi, dan landmark (opsional)

Notebook eICU tidak menyimpan prediksi ke file. Tambahkan potongan ini **di akhir CELL 10** notebook
`COST_eICU_EXTERNAL_FINAL_CORRECTED.ipynb`, jalankan ulang (inference saja), lalu jalankan sel di bawah:

```python
_out = ROOT / "mimic-iv-2.1/experiment_outputs/external_eicu/eicu_external_predictions.npz"
_extra = {c: cohort_e[c].to_numpy() for c in ["unitdischargeoffset", "hospitaldischargeoffset"] if c in cohort_e.columns}
np.savez_compressed(_out, y=y, prob=y_prob_ext, patientunitstayid=final_ids, **_extra)
print("saved", _out)
```

In [ ]:
# CELL 13 — A8: eICU EXTERNAL CALIBRATION (runs only if the prediction file exists)
if RUN_EICU and EICU_PRED.exists():
    e = np.load(EICU_PRED, allow_pickle=True); ye, pe = e["y"].astype(int), e["prob"].astype(float)
    rows = []
    def addr(nm, p, m=None):
        m = np.ones(len(ye), bool) if m is None else m
        r = {"eICU analysis": nm, "N": int(m.sum()), "AUROC": roc_auc_score(ye[m], p[m])}; r.update(calib_stats(ye[m], p[m])); rows.append(r)
    addr("As reported (frozen model)", pe)
    addr("Prior-shift with MIMIC training prevalence", prior_shift(pe, train_prev))
    if RUN_CHECKPOINT_INFERENCE:
        addr("Platt fitted on MIMIC validation (transported)", platt_main.predict(pe))
    st, _ = crossfit_recal(ye, pe, InterceptOnly); r = {"eICU analysis": "Recalibration-in-the-large on eICU (cross-fitted)", "N": len(ye), "AUROC": roc_auc_score(ye, pe)}; r.update(st); rows.append(r)
    st, _ = crossfit_recal(ye, pe, Platt);         r = {"eICU analysis": "Platt on eICU (cross-fitted)", "N": len(ye), "AUROC": roc_auc_score(ye, pe)}; r.update(st); rows.append(r)
    if "hospitaldischargeoffset" in e.files:
        died_lt48 = (ye == 1) & (e["hospitaldischargeoffset"].astype(float) < 48 * 60)
        addr("Landmark: exclude deaths < 48 h", pe, ~died_lt48)
        REPORT["eicu_deaths_lt48"] = int(died_lt48.sum())
    if "unitdischargeoffset" in e.files:
        out_lt48 = e["unitdischargeoffset"].astype(float) < 48 * 60
        REPORT["eicu_unit_discharge_lt48"] = int(out_lt48.sum())
    eicu_tab = save_table(pd.DataFrame(rows), "A8_eicu_calibration")
    show(eicu_tab.round(4))
    reliability_plot({"As reported": pe, "Prior-shift": prior_shift(pe, train_prev)}, ye, "eICU-CRD external cohort", "A8_reliability_eicu.png")
    REPORT["eicu_table"] = eicu_tab.round(4).to_dict(orient="records")
else:
    print("A8 skipped — eICU prediction file not found:", EICU_PRED)